# Kaggle House Prices — Complete Pipeline

Notebook sạch theo `GUIDE_HANDSON_NOTEBOOK.md`.

**Metric:** RMSLE (càng thấp càng tốt)  
**Mục tiêu cuối:** `submission.csv` (`Id`, `SalePrice`)

| Phase | Nội dung | Status |
|------:|----------|--------|
| 0 | Setup & hiểu bài toán | done |
| 1 | Load data sạch | done |
| 2 | EDA | done |
| 3 | Missing / cleaning | done |
| 4 | Feature engineering | **đang làm** |
| 5 | Encoding & scaling | pending |
| 6 | Model + validation (CV / RMSLE) | pending |
| 7 | Predict test + submission | pending |
| 8 | Error analysis (optional) | pending |

> Kernel khuyến nghị: **Python (.venv House Prices)**

## 0. Setup & imports

Kiểm tra đang chạy đúng `.venv` của project trước khi load data.

In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

assert ".venv" in sys.executable.replace("\\", "/"), (
    "Sai kernel! Chọn kernel Python (.venv House Prices) hoặc .venv/bin/python"
)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import norm, probplot

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path(".")
print("Imports OK")

## 1. Load data

- Load **local** `train.csv` / `test.csv` (không wget URL GitHub `/blob/`).
- Giữ `test_ids` trước khi xử lý.
- Tách target `y`; không để `SalePrice` lẫn vào feature của test.

In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

assert train_path.exists(), f"Không thấy {train_path.resolve()}"
assert test_path.exists(), f"Không thấy {test_path.resolve()}"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("train:", train.shape)
print("test :", test.shape)

assert train.shape == (1460, 81), f"train shape lệch: {train.shape}"
assert test.shape == (1459, 80), f"test shape lệch: {test.shape}"
assert "SalePrice" in train.columns
assert "SalePrice" not in test.columns

In [ ]:
# Giữ Id của test để submission sau này
test_ids = test["Id"].copy()

# Target (thang gốc). Modeling sẽ dùng log1p ở Phase 2/6.
y = train["SalePrice"].copy()

# Feature matrix: cùng cột giữa train/test (bỏ SalePrice khỏi train features)
X_train_raw = train.drop(columns=["SalePrice"]).copy()
X_test_raw = test.copy()

print("test_ids:", len(test_ids), "| range:", test_ids.min(), "→", test_ids.max())
print("y describe:")
display(y.describe())
print("X_train_raw:", X_train_raw.shape, "| X_test_raw:", X_test_raw.shape)
assert list(X_train_raw.columns) == list(X_test_raw.columns)

In [ ]:
train.head()

### Phase 1 — Definition of Done

- [x] Load `train.csv` / `test.csv` local thành công
- [x] Có `test_ids`
- [x] Có `y`, `X_train_raw`, `X_test_raw` cùng schema
- [x] Chạy lại toàn bộ cell phía trên không lỗi

Tiếp theo → **Phase 2: EDA**.

## 2. EDA

Khám phá để **quyết định preprocessing**, không chỉ vẽ chart.

Checklist Phase 2:
- Tổng quan dtype / missing train vs test
- Phân phối `SalePrice` → quyết định `log1p`
- Outlier `GrLivArea` (chỉ trên train)
- Corr numeric + vài categorical/ordinal quan trọng
- Ghi chú quyết định cho Phase 3–5

### 2.1 Tổng quan dữ liệu

In [ ]:
# Dtype overview
num_cols = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train_raw.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric columns : {len(num_cols)}")
print(f"Categorical cols: {len(cat_cols)}")
print("\ntrain.info():")
train.info()
display(train.describe().T.head(20))

In [ ]:
def missing_rate(df: pd.DataFrame) -> pd.Series:
    return (df.isna().sum() / len(df) * 100).sort_values(ascending=False)

miss_train = missing_rate(X_train_raw)
miss_test = missing_rate(X_test_raw)

missing_cmp = pd.DataFrame({
    "train_%": miss_train,
    "test_%": miss_test,
}).fillna(0)
missing_cmp["max_%"] = missing_cmp.max(axis=1)
missing_cmp = missing_cmp[missing_cmp["max_%"] > 0].sort_values("max_%", ascending=False)

print(f"Cột có missing: {len(missing_cmp)}")
display(missing_cmp.head(25))

# NA mang nghĩa "không có tiện ích" theo data_description (không phải missing ngẫu nhiên)
na_means_none = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
print("\nTop missing thuộc nhóm 'NA = không có':")
display(missing_cmp.loc[missing_cmp.index.intersection(na_means_none)])

In [ ]:
# Visual: top missing train vs test
top_n = missing_cmp.head(15)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top_n))
w = 0.38
ax.bar(x - w/2, top_n["train_%"], width=w, label="train")
ax.bar(x + w/2, top_n["test_%"], width=w, label="test")
ax.set_xticks(x)
ax.set_xticklabels(top_n.index, rotation=45, ha="right")
ax.set_ylabel("Missing %")
ax.set_title("Missing rate: train vs test (top 15)")
ax.legend()
plt.tight_layout()
plt.show()

print("Insight: PoolQC/MiscFeature/Alley/Fence missing rất cao → thường là 'không có', nên fill 'None' thay vì drop cột ngay.")

### 2.2 Phân phối target `SalePrice`

In [ ]:
y_log = np.log1p(y)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Raw
sns.histplot(y, kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title(f"SalePrice | skew={y.skew():.3f}, kurt={y.kurt():.3f}")

probplot(y, plot=axes[0, 1])
axes[0, 1].set_title("QQ — SalePrice (raw)")

# Log
sns.histplot(y_log, kde=True, ax=axes[1, 0], color="darkorange")
axes[1, 0].set_title(f"log1p(SalePrice) | skew={y_log.skew():.3f}, kurt={y_log.kurt():.3f}")

probplot(y_log, plot=axes[1, 1])
axes[1, 1].set_title("QQ — log1p(SalePrice)")

plt.tight_layout()
plt.show()

print("Quyết định: dùng y_log = np.log1p(y) khi train model; predict xong phải np.expm1 trước submit.")

### 2.3 Outliers — `GrLivArea` vs `SalePrice`

Chỉ xem xét drop trên **train**. Không xóa outlier trên test.

In [ ]:
outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300_000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(train["GrLivArea"], train["SalePrice"], alpha=0.5, s=18)
axes[0].scatter(train.loc[outlier_mask, "GrLivArea"], train.loc[outlier_mask, "SalePrice"],
                color="red", s=40, label="candidate outliers")
axes[0].axvline(4000, color="gray", ls="--", lw=1)
axes[0].set_xlabel("GrLivArea")
axes[0].set_ylabel("SalePrice")
axes[0].set_title("Trước khi drop")
axes[0].legend()

display(train.loc[outlier_mask, ["Id", "GrLivArea", "OverallQual", "SalePrice"]])
print(f"Số outlier ứng viên: {outlier_mask.sum()}")

# Áp dụng drop trên train only (giữ biến gốc nếu cần so sánh)
train_eda = train.loc[~outlier_mask].copy()
y_eda = train_eda["SalePrice"].copy()
y_log_eda = np.log1p(y_eda)
X_train_eda = train_eda.drop(columns=["SalePrice"]).copy()

axes[1].scatter(train_eda["GrLivArea"], train_eda["SalePrice"], alpha=0.5, s=18)
axes[1].set_xlabel("GrLivArea")
axes[1].set_ylabel("SalePrice")
axes[1].set_title(f"Sau drop ({len(train_eda)} rows)")
plt.tight_layout()
plt.show()

print("Quyết định: drop 2 điểm GrLivArea>4000 & SalePrice<300k trên train (diện tích lớn nhưng giá thấp — nhiễu cho linear/GBM).")
print(f"train: {len(train)} → {len(train_eda)} | test giữ nguyên: {len(test)}")

### 2.4 Quan hệ feature–target

In [ ]:
# Correlation với SalePrice (numeric, full train_eda — không sample 25%)
corr_target = (
    train_eda.select_dtypes(include=["number"])
    .corr(numeric_only=True)["SalePrice"]
    .drop("SalePrice")
    .sort_values(key=np.abs, ascending=False)
)

print("Top |corr| với SalePrice:")
display(corr_target.head(20).to_frame("corr_SalePrice"))

top_feats = corr_target.head(12).index.tolist()
corr_mat = train_eda[top_feats + ["SalePrice"]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Heatmap — top numeric features + SalePrice")
plt.tight_layout()
plt.show()

print("Insight: OverallQual, GrLivArea, GarageCars/Area, TotalBsmtSF, 1stFlrSF là tín hiệu mạnh.")
print("Lưu ý đa cộng tuyến: GarageCars↔GarageArea, TotalBsmtSF↔1stFlrSF — cân nhắc TotalSF ở Phase 4.")

In [ ]:
# Ordinal / quality-like features
ordinal_like = ["OverallQual", "ExterQual", "KitchenQual", "BsmtQual", "GarageFinish", "FireplaceQu"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(ordinal_like):
    order = train_eda[col].value_counts().index if train_eda[col].dtype == object else sorted(train_eda[col].dropna().unique())
    sns.boxplot(data=train_eda, x=col, y="SalePrice", order=order, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("Insight: chất lượng càng cao (Ex/Gd, OverallQual cao) → SalePrice tăng rõ — phù hợp ordinal encoding.")

In [ ]:
# Categorical: Neighborhood & MSZoning — median price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nb_order = train_eda.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).index
sns.boxplot(data=train_eda, x="Neighborhood", y="SalePrice", order=nb_order, ax=axes[0])
axes[0].tick_params(axis="x", rotation=90)
axes[0].set_title("SalePrice by Neighborhood (median desc)")

sns.boxplot(data=train_eda, x="MSZoning", y="SalePrice", ax=axes[1])
axes[1].set_title("SalePrice by MSZoning")
plt.tight_layout()
plt.show()

print("Insight: Neighborhood khác biệt mạnh → giữ làm categorical (one-hot / target-aware sau này).")

### 2.5 Ghi chú quyết định preprocessing (cho Phase 3–5)

Quyết định rút từ EDA — Phase 3 sẽ implement theo đúng list này.

#### Imputation
| Nhóm cột | Cách fill | Lý do |
|----------|-----------|-------|
| `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu` | `"None"` | NA = không có tiện ích |
| `GarageType/Finish/Qual/Cond` | `"None"` | NA = không garage |
| `GarageYrBlt`, `GarageCars`, `GarageArea` (nếu NA) | `0` | Không garage |
| `BsmtQual/Cond/Exposure/FinType*` | `"None"` | NA = không basement |
| `Bsmt*` numeric liên quan (nếu NA trên test) | `0` | Không basement |
| `MasVnrType` | `"None"` | Thường đi kèm không veneer |
| `MasVnrArea` | `0` | |
| `LotFrontage` | **median theo `Neighborhood`** (fit train) | Tương quan theo khu |
| Còn lại categorical | mode (fit train) | Missing thật, ít |
| Còn lại numeric | median (fit train) | An toàn, chống outlier |

#### Encoding
- **Ordinal** (có thứ tự chất lượng): `ExterQual/Cond`, `BsmtQual/Cond`, `HeatingQC`, `KitchenQual`, `FireplaceQu`, `GarageQual/Cond`, `PoolQC`, `BsmtExposure`, `BsmtFinType*`, `GarageFinish`, `Functional`, `Fence`, `LandSlope`, …
- **Nominal** (one-hot, `handle_unknown='ignore'`): `MSZoning`, `Neighborhood`, `Condition*`, `HouseStyle`, `Exterior*`, `SaleType`, `SaleCondition`, …

#### Feature engineering (Phase 4)
- `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`
- `TotalBath = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath`
- `Age = YrSold - YearBuilt`
- `RemodAge = YrSold - YearRemodAdd`
- Flags: `HasGarage`, `HasBsmt`, `HasPool`

#### Có thể bỏ / thận trọng
- `Id` — không dùng làm feature (nhưng giữ `test_ids` để submit)
- `Utilities` — gần như hằng trên train → cân nhắc drop
- `PoolQC` / `MiscFeature` — rất sparse; vẫn có thể giữ sau khi fill `"None"`

#### Target & outliers
- Train trên `y_log = log1p(SalePrice)`; submit bằng `expm1`
- Drop 2 outlier `GrLivArea > 4000 & SalePrice < 300000` **chỉ trên train**
- Biến dùng tiếp: `train_eda`, `X_train_eda`, `y_eda`, `y_log_eda`

### Phase 2 — Definition of Done

- [x] Tổng quan dtype + missing train vs test
- [x] Phân phối SalePrice / log1p + quyết định dùng `y_log`
- [x] Xử lý outlier `GrLivArea` trên train only
- [x] Corr numeric + boxplot ordinal/categorical có insight
- [x] Ghi chú quyết định imputation / encoding / FE

Tiếp theo → **Phase 3: Missing values & cleaning** (fit train → transform test).

## 3. Missing values & cleaning

Nguyên tắc: **fit trên train → transform train & test** (tránh leakage).

- NA mang nghĩa “không có” → `"None"` / `0`
- `LotFrontage` → median theo `Neighborhood` (học từ train)
- Còn lại: mode (cat) / median (num) — thống kê từ train
- Drop `Id` khỏi feature (đã giữ `test_ids`)
- Ordinal map + one-hot để **Phase 5**; FE để **Phase 4**

### 3.1 Helper: MissingCleaner (fit / transform)

In [ ]:
class MissingCleaner:
    """Impute missing values with statistics learned from train only."""

    CAT_NONE = [
        "Alley", "MasVnrType", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu", "GarageType",
        "GarageFinish", "GarageQual", "GarageCond", "PoolQC", "Fence",
        "MiscFeature",
    ]
    NUM_ZERO = [
        "MasVnrArea", "GarageYrBlt", "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath",
    ]

    def __init__(self, drop_cols=None):
        self.drop_cols = list(drop_cols or ["Id"])
        self.lotfrontage_medians_ = None
        self.lotfrontage_global_ = None
        self.cat_modes_ = {}
        self.num_medians_ = {}
        self.columns_ = None

    def _apply_special(self, out: pd.DataFrame) -> pd.DataFrame:
        for c in self.CAT_NONE:
            if c in out.columns:
                out[c] = out[c].fillna("None")
        for c in self.NUM_ZERO:
            if c in out.columns:
                out[c] = out[c].fillna(0)
        if (
            self.lotfrontage_medians_ is not None
            and "LotFrontage" in out.columns
            and "Neighborhood" in out.columns
        ):
            med = self.lotfrontage_medians_
            glob = self.lotfrontage_global_
            out["LotFrontage"] = [
                (v if pd.notna(v) else med.get(nb, glob))
                for v, nb in zip(out["LotFrontage"], out["Neighborhood"])
            ]
        return out

    def fit(self, X: pd.DataFrame):
        df = X.copy()
        self.columns_ = [c for c in df.columns if c not in self.drop_cols]

        if "LotFrontage" in df.columns and "Neighborhood" in df.columns:
            self.lotfrontage_medians_ = df.groupby("Neighborhood")["LotFrontage"].median()
            self.lotfrontage_global_ = float(df["LotFrontage"].median())

        # Learn fallback stats on train AFTER special fills (covers test-only missing)
        work = self._apply_special(df[self.columns_].copy())
        self.cat_modes_ = {}
        self.num_medians_ = {}
        for c in work.columns:
            if pd.api.types.is_numeric_dtype(work[c]):
                self.num_medians_[c] = float(work[c].median())
            else:
                mode = work[c].mode(dropna=True)
                self.cat_modes_[c] = mode.iloc[0] if len(mode) else "None"
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        out = X.reindex(columns=self.columns_).copy()
        out = self._apply_special(out)

        for c, mode in self.cat_modes_.items():
            if c in out.columns:
                out[c] = out[c].fillna(mode)
        for c, med in self.num_medians_.items():
            if c in out.columns:
                out[c] = out[c].fillna(med)
        return out

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


print("MissingCleaner defined")


### 3.2 Fit trên `X_train_eda` → transform train & test

In [ ]:
# Utilities gần như hằng trên train → cân nhắc drop (theo Phase 2 notes)
print("Utilities value_counts (train_eda):")
display(X_train_eda["Utilities"].value_counts(dropna=False))

cleaner = MissingCleaner(drop_cols=["Id", "Utilities"])
X_train_clean = cleaner.fit_transform(X_train_eda)
X_test_clean = cleaner.transform(X_test_raw)

print("X_train_clean:", X_train_clean.shape)
print("X_test_clean :", X_test_clean.shape)
assert list(X_train_clean.columns) == list(X_test_clean.columns)

In [ ]:
# Kiểm tra không còn NaN
na_train = int(X_train_clean.isna().sum().sum())
na_test = int(X_test_clean.isna().sum().sum())
print(f"NaN còn lại — train: {na_train} | test: {na_test}")

if na_train or na_test:
    print("Chi tiết train:")
    display(X_train_clean.isna().sum()[X_train_clean.isna().sum() > 0])
    print("Chi tiết test:")
    display(X_test_clean.isna().sum()[X_test_clean.isna().sum() > 0])

assert na_train == 0 and na_test == 0, "Vẫn còn missing — kiểm tra lại cleaner"

# Target gắn với train_eda (đã drop outlier)
y_clean = y_eda.copy()
y_log_clean = y_log_eda.copy()
assert len(X_train_clean) == len(y_clean) == len(y_log_clean)
print("Target sync OK:", len(y_clean), "rows")

In [ ]:
# Sanity: vài cột đã fill 'None' / 0
check_cols = ["PoolQC", "GarageType", "BsmtQual", "MasVnrType", "MasVnrArea", "GarageYrBlt", "LotFrontage"]
summary = pd.DataFrame({
    "dtype": X_train_clean[check_cols].dtypes.astype(str),
    "nunique": X_train_clean[check_cols].nunique(),
    "has_None": [(X_train_clean[c] == "None").sum() if not pd.api.types.is_numeric_dtype(X_train_clean[c]) else 0 for c in check_cols],
    "zeros": [(X_train_clean[c] == 0).sum() if pd.api.types.is_numeric_dtype(X_train_clean[c]) else 0 for c in check_cols],
}, index=check_cols)
display(summary)

display(X_train_clean.head())

### Phase 3 — Definition of Done

- [x] Impute không dùng `fillna(mode Series)` / `fillna(df[col]==0)`
- [x] NA “không có tiện ích” → `"None"`; numeric liên quan → `0`
- [x] `LotFrontage` median theo `Neighborhood` (fit train)
- [x] Mode/median phần còn lại học từ train
- [x] `X_train_clean` / `X_test_clean` cùng cột, **0 NaN**
- [x] `y_clean` / `y_log_clean` khớp số dòng train sau drop outlier

**Biến dùng tiếp:** `X_train_clean`, `X_test_clean`, `y_clean`, `y_log_clean`, `cleaner`

Tiếp theo → **Phase 4: Feature engineering** (`TotalSF`, `TotalBath`, `Age`, …).

## 4. Feature engineering

Tạo feature mới **deterministic** (cùng công thức train/test, không leakage).

Tối thiểu theo guide:
- `TotalSF`, `TotalBath`, `Age`, `RemodAge`
- Flags: `HasGarage`, `HasBsmt`, `HasPool`
- (thêm) `HasFireplace`, `TotalPorchSF` — hữu ích, ít rủi ro

### 4.1 FeatureEngineer

In [21]:
class FeatureEngineer:
    """Create derived features. Pure transform — no train statistics."""

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()

        # Area aggregates
        df["TotalSF"] = (
            df["TotalBsmtSF"].astype(float)
            + df["1stFlrSF"].astype(float)
            + df["2ndFlrSF"].astype(float)
        )
        df["TotalPorchSF"] = (
            df["OpenPorchSF"].astype(float)
            + df["EnclosedPorch"].astype(float)
            + df["3SsnPorch"].astype(float)
            + df["ScreenPorch"].astype(float)
        )

        # Bathrooms
        df["TotalBath"] = (
            df["FullBath"].astype(float)
            + 0.5 * df["HalfBath"].astype(float)
            + df["BsmtFullBath"].astype(float)
            + 0.5 * df["BsmtHalfBath"].astype(float)
        )

        # Age-like (YrSold as reference year in dataset)
        df["Age"] = df["YrSold"].astype(float) - df["YearBuilt"].astype(float)
        df["RemodAge"] = df["YrSold"].astype(float) - df["YearRemodAdd"].astype(float)
        df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

        # Clip weird negatives (remodels before built / data glitches)
        df["Age"] = df["Age"].clip(lower=0)
        df["RemodAge"] = df["RemodAge"].clip(lower=0)

        # Presence flags
        df["HasGarage"] = (df["GarageArea"].astype(float) > 0).astype(int)
        df["HasBsmt"] = (df["TotalBsmtSF"].astype(float) > 0).astype(int)
        df["HasPool"] = (df["PoolArea"].astype(float) > 0).astype(int)
        df["HasFireplace"] = (df["Fireplaces"].astype(float) > 0).astype(int)
        df["Has2ndFlr"] = (df["2ndFlrSF"].astype(float) > 0).astype(int)

        return df

    def fit(self, X: pd.DataFrame):
        return self

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


print("FeatureEngineer defined")

FeatureEngineer defined


### 4.2 Áp dụng lên train & test đã clean

In [22]:
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train_clean)
X_test_fe = fe.transform(X_test_clean)

new_cols = [c for c in X_train_fe.columns if c not in X_train_clean.columns]
print("Feature mới:", new_cols)
print("X_train_fe:", X_train_fe.shape, "| X_test_fe:", X_test_fe.shape)
assert list(X_train_fe.columns) == list(X_test_fe.columns)
assert X_train_fe.isna().sum().sum() == 0 and X_test_fe.isna().sum().sum() == 0

Feature mới: ['TotalSF', 'TotalPorchSF', 'TotalBath', 'Age', 'RemodAge', 'IsRemodeled', 'HasGarage', 'HasBsmt', 'HasPool', 'HasFireplace', 'Has2ndFlr']
X_train_fe: (1458, 89) | X_test_fe: (1459, 89)


In [23]:
# Sanity checks: no NaN/Inf, ages non-negative
fe_num = ["TotalSF", "TotalBath", "Age", "RemodAge", "TotalPorchSF"]
display(X_train_fe[fe_num + ["HasGarage", "HasBsmt", "HasPool", "HasFireplace", "IsRemodeled"]].describe().T)

bad = []
for c in fe_num:
    if (X_train_fe[c] < 0).any() or (X_test_fe[c] < 0).any():
        bad.append(c)
    if np.isinf(X_train_fe[c]).any() or np.isinf(X_test_fe[c]).any():
        bad.append(c + "(inf)")

assert not bad, f"Feature lỗi: {bad}"
print("Sanity OK — không âm / Inf trên các feature chính")

# Nhanh: corr feature mới với target (log)
tmp = X_train_fe[new_cols].copy()
tmp["y_log"] = y_log_clean.values
corr_new = tmp.corr(numeric_only=True)["y_log"].drop("y_log").sort_values(key=np.abs, ascending=False)
print("\n|corr| với y_log (feature mới):")
display(corr_new.to_frame("corr_y_log"))

,count,mean,std,min,25%,50%,75%,max
TotalSF,"1,458.0000","2,557.1502",774.1098,334.0000,"2,008.5000","2,473.0000","3,002.2500","6,872.0000"
TotalBath,"1,458.0000",2.2075,0.7813,1.0000,2.0000,2.0000,2.5000,6.0000
Age,"1,458.0000",36.5981,30.2406,0.0000,8.0000,35.0000,54.0000,136.0000
RemodAge,"1,458.0000",22.9822,20.6365,0.0000,4.0000,14.0000,41.0000,60.0000
TotalPorchSF,"1,458.0000",86.7257,104.7924,0.0000,0.0000,48.0000,135.7500,"1,027.0000"
HasGarage,"1,458.0000",0.9444,0.2291,0.0000,1.0000,1.0000,1.0000,1.0000
HasBsmt,"1,458.0000",0.9746,0.1573,0.0000,1.0000,1.0000,1.0000,1.0000
HasPool,"1,458.0000",0.0041,0.0640,0.0000,0.0000,0.0000,0.0000,1.0000
HasFireplace,"1,458.0000",0.5267,0.4995,0.0000,0.0000,1.0000,1.0000,1.0000
IsRemodeled,"1,458.0000",0.4767,0.4996,0.0000,0.0000,0.0000,1.0000,1.0000


Sanity OK — không âm / Inf trên các feature chính

|corr| với y_log (feature mới):


,corr_y_log
TotalSF,0.8253
TotalBath,0.6767
Age,-0.5878
RemodAge,-0.5685
HasFireplace,0.5103
HasGarage,0.3230
HasBsmt,0.1996
TotalPorchSF,0.1956
Has2ndFlr,0.1506
HasPool,0.0765


### Phase 4 — Definition of Done

- [x] `TotalSF`, `TotalBath`, `Age`, `RemodAge` (+ porch / remodel flag)
- [x] Flags `HasGarage`, `HasBsmt`, `HasPool` (+ fireplace / 2nd floor)
- [x] Train & test cùng schema; không NaN/Inf; age đã clip ≥ 0

**Biến dùng tiếp:** `X_train_fe`, `X_test_fe`, `fe`  
(target vẫn: `y_clean`, `y_log_clean`)

Tiếp theo → **Phase 5: Encoding & scaling** (ordinal + one-hot, align cột).

## 5. Encoding & scaling

> TODO Phase 5: ordinal map / one-hot align cột train–test; scaler cho linear models.

In [ ]:
# TODO Phase 5
pass

## 6. Model training & validation

> TODO Phase 6: CV + RMSLE; baseline → Ridge/Lasso → RF/GBM; bảng so sánh.

In [ ]:
def rmsle(y_true, y_pred):
    """RMSLE trên thang giá gốc."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(y_pred, 0, None)))))

# TODO Phase 6: train models + CV
pass

## 7. Predict test + submission

> TODO Phase 7: fit full train → predict test → `expm1` nếu dùng log → `submission.csv`.

In [ ]:
# TODO Phase 7
# submission = pd.DataFrame({"Id": test_ids, "SalePrice": pred_prices})
# submission.to_csv("submission.csv", index=False)
pass

## 8. (Optional) Error analysis & next experiments

> TODO Phase 8: residual plots, feature importance, ý tưởng cải thiện tiếp.

In [ ]:
# TODO Phase 8
pass